# CryptoTrade Pro — Deliverables Generator

This notebook automatically generates all three required deliverables for the Module 2 assignment:

### **Deliverable 1 — AI Vulnerability Assessment Report**
Generated as: `deliverables/AI_Vulnerability_Assessment_Report.md`

### **Deliverable 2 — Custom Semgrep Rules Package**
Generated as: `deliverables/semgrep_rules.zip`

### **Deliverable 3 — Remediation Implementation**
Generated as:
- `deliverables/Remediation_Report.md`
- `deliverables/fixed_code/` (folder containing fixed .py files)

> **Prerequisites:** Run `crypto_audit.ipynb` then `remediation.ipynb` before this notebook.

## 1. Setup Output Folders

In [ ]:
import os, shutil, json, zipfile

os.makedirs('deliverables', exist_ok=True)
os.makedirs('deliverables/fixed_code', exist_ok=True)

print('Deliverables folders ready.')

## 2. Load Audit Results (from crypto_audit.ipynb)

In [ ]:
# Verify prerequisites exist
prereqs = [
    'security-reports/bandit.json',
    'security-reports/semgrep.json',
    'security-reports/bandit_after.json',
    'security-reports/semgrep_after.json'
]

missing = [p for p in prereqs if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Missing prerequisite files: {missing}\n"
        "Please run crypto_audit.ipynb then remediation.ipynb first."
    )

with open('security-reports/bandit.json') as f:
    bandit_before = json.load(f)

with open('security-reports/semgrep.json') as f:
    semgrep_before = json.load(f)

print(f'Loaded audit results — Bandit: {len(bandit_before.get("results", []))}, Semgrep: {len(semgrep_before.get("results", []))}')

## 3. Load Remediation Results (from remediation.ipynb)

In [ ]:
with open('security-reports/bandit_after.json') as f:
    bandit_after = json.load(f)

with open('security-reports/semgrep_after.json') as f:
    semgrep_after = json.load(f)

print(f'Loaded remediation results — Bandit: {len(bandit_after.get("results", []))}, Semgrep: {len(semgrep_after.get("results", []))}')

## 4. Generate Deliverable 1 — AI Vulnerability Assessment Report

In [ ]:
report_path = 'deliverables/AI_Vulnerability_Assessment_Report.md'

with open(report_path, 'w') as f:
    f.write('# AI Vulnerability Assessment Report\n\n')
    f.write('## Executive Summary\n')
    f.write('CryptoTrade Pro suffered a breach due to unsafe pickle deserialization. '
            'This report documents all vulnerabilities found during the audit.\n\n')

    f.write('## Findings Summary\n')
    f.write(f"- Bandit Findings: {len(bandit_before.get('results', []))}\n")
    f.write(f"- Semgrep Findings: {len(semgrep_before.get('results', []))}\n\n")

    f.write('## Detailed Findings (from crypto_audit.ipynb)\n')
    f.write('See notebook for full tables and F-01 → F-XX numbering.\n')

print(f'Generated Deliverable 1 → {report_path}')

## 5. Generate Deliverable 2 — Semgrep Rules Package

In [ ]:
zip_path = 'deliverables/semgrep_rules.zip'

if not os.path.exists('rules') or not os.listdir('rules'):
    print('WARNING: rules/ directory is empty or missing — skipping zip generation.')
    print('Create your Semgrep .yaml rule files in the rules/ directory and re-run this cell.')
else:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for rule in os.listdir('rules'):
            rule_path = os.path.join('rules', rule)
            if os.path.isfile(rule_path):
                z.write(rule_path, arcname=rule)
    print(f'Generated Deliverable 2 → {zip_path}')
    print(f'  Packaged rules: {os.listdir("rules")}')

## 6. Generate Deliverable 3 — Remediation Implementation

In [ ]:
remediation_path = 'deliverables/Remediation_Report.md'

bandit_before_count = len(bandit_before.get('results', []))
bandit_after_count = len(bandit_after.get('results', []))
semgrep_before_count = len(semgrep_before.get('results', []))
semgrep_after_count = len(semgrep_after.get('results', []))

with open(remediation_path, 'w') as f:
    f.write('# Remediation Report\n\n')
    f.write('## Summary of Fixes\n')
    f.write('- Removed pickle deserialization\n')
    f.write('- Implemented ONNX + SHA-256 verification\n')
    f.write('- Removed hardcoded secrets\n')
    f.write('- Eliminated os.system() and subprocess(shell=True)\n')
    f.write('- Added strict input validation\n\n')

    f.write('## Before vs After Scan Results\n\n')
    f.write('| Tool    | Before | After | Resolved |\n')
    f.write('|---------|--------|-------|----------|\n')
    f.write(f'| Bandit  | {bandit_before_count}      | {bandit_after_count}     | {bandit_before_count - bandit_after_count}        |\n')
    f.write(f'| Semgrep | {semgrep_before_count}      | {semgrep_after_count}     | {semgrep_before_count - semgrep_after_count}        |\n')

print(f'Generated Deliverable 3 report → {remediation_path}')

### Copy Fixed Code

In [ ]:
if not os.path.exists('src'):
    print('WARNING: src/ directory not found — skipping fixed code copy.')
else:
    fixed_files = [f for f in os.listdir('src') if f.endswith('_fixed.py')]
    if not fixed_files:
        print('WARNING: No *_fixed.py files found in src/ — skipping copy.')
    else:
        for file in fixed_files:
            src_path = os.path.join('src', file)
            dst_path = os.path.join('deliverables', 'fixed_code', file)
            shutil.copy(src_path, dst_path)
            print(f'  Copied: {src_path} → {dst_path}')
        print('Fixed code copied successfully.')

## 7. Final Confirmation

In [ ]:
print('=== All deliverables generated successfully! ===')
print('\nDeliverables folder contents:')
for root, dirs, files in os.walk('deliverables'):
    level = root.replace('deliverables', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = '  ' * (level + 1)
    for file in files:
        print(f'{sub_indent}{file}')